# Renewable Energy Sources in India
## Data Science Project — Google Colab

This notebook implements the five project chapters: introduction, preprocessing and EDA, Git/GitHub workflow, model development, and results/conclusion.

**Goal:** estimate state/region-level total installed renewable capacity one year ahead using historical state-year records. The five technology capacities are added to create the target. Capacity values are retained in the dataset's original unit because the CSV does not document whether they are MW or GW.

## Chapter 2 — Install libraries and load the dataset
Run the install cell, then upload the original CSV file when prompted.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn joblib

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Upload the renewable energy CSV dataset.")
df_raw = pd.read_csv(csv_files[0])
print("Dataset shape:", df_raw.shape)
display(df_raw.head())

## Chapter 2 — Data understanding and quality checks
Check the structure, missing values, duplicates, repeated State–Year pairs and negative values. The target is the sum of the five installed-capacity columns.

In [ ]:
CAPACITY_COLUMNS = [
    "Installed - Bio-Mass Power",
    "Installed - Small Hydro Power",
    "Installed - Solar Power",
    "Installed - Wind Power",
    "Installed - Waste to Energy",
]
TARGET = "Total_Renewable_Capacity"
required = ["State", "Year", "Population", "GSDP (crore INR)", *CAPACITY_COLUMNS]
missing_columns = [c for c in required if c not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

numeric_cols = df_raw.select_dtypes(include=np.number).columns
print("Missing cells:", int(df_raw.isna().sum().sum()))
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("Repeated State-Year pairs:", int(df_raw.duplicated(["State", "Year"]).sum()))
print("Negative numeric values:", int((df_raw[numeric_cols] < 0).sum().sum()))
print("States/regions:", df_raw["State"].nunique())
print("Year range:", int(df_raw["Year"].min()), "to", int(df_raw["Year"].max()))

df = df_raw.copy()
df[TARGET] = df[CAPACITY_COLUMNS].sum(axis=1)
display(df.describe(include="all").T)

## Chapter 2 — Exploratory Data Analysis (EDA)
The charts describe the annual trend, the contribution of each technology, the leading states/regions and the distribution of total capacity. They are descriptive, not forecasts.

In [ ]:
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

annual = df.groupby("Year")[TARGET].sum()
plt.figure(figsize=(10, 4))
plt.plot(annual.index, annual.values, marker="o")
plt.title("Total Installed Renewable Capacity by Year")
plt.xlabel("Year"); plt.ylabel("Capacity (dataset units)"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(PLOTS_DIR / "annual_capacity_trend.png", dpi=150); plt.show()

technology = df[CAPACITY_COLUMNS].sum().sort_values(ascending=False)
plt.figure(figsize=(9, 4))
plt.bar(technology.index, technology.values)
plt.title("Installed Capacity by Renewable Technology")
plt.ylabel("Capacity (dataset units)"); plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "capacity_by_technology.png", dpi=150); plt.show()

latest_year = int(df["Year"].max())
top_states = df[df["Year"] == latest_year].nlargest(10, TARGET).sort_values(TARGET)
plt.figure(figsize=(9, 5))
plt.barh(top_states["State"], top_states[TARGET])
plt.title(f"Top States/Regions by Capacity ({latest_year})")
plt.xlabel("Capacity (dataset units)")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "top_states_latest_year.png", dpi=150); plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df[TARGET], bins=30, edgecolor="black")
plt.title("Distribution of Total Renewable Capacity")
plt.xlabel("Capacity (dataset units)"); plt.ylabel("Number of records")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "target_distribution.png", dpi=150); plt.show()

## Chapter 2 — Preprocessing and feature engineering
Sort by state and year, create previous-year technology capacities and previous-year total capacity, then remove each state's first observation because it has no previous-year input. Current-year technology values are excluded from model inputs to avoid target leakage.

In [ ]:
LAG_COLUMNS = [f"Previous Year - {c}" for c in CAPACITY_COLUMNS]
PREVIOUS_TOTAL = "Previous Total Capacity"
model_df = df.sort_values(["State", "Year"]).copy()
for source_col, lag_col in zip(CAPACITY_COLUMNS, LAG_COLUMNS):
    model_df[lag_col] = model_df.groupby("State")[source_col].shift(1)
model_df[PREVIOUS_TOTAL] = model_df.groupby("State")[TARGET].shift(1)
model_df["Capacity Change"] = model_df[TARGET] - model_df[PREVIOUS_TOTAL]
model_df = model_df.dropna(subset=[*LAG_COLUMNS, PREVIOUS_TOTAL]).copy()

FEATURES = ["State", "Year", "Population", "GSDP (crore INR)", PREVIOUS_TOTAL, *LAG_COLUMNS]
NUMERIC_FEATURES = ["Year", "Population", "GSDP (crore INR)", PREVIOUS_TOTAL, *LAG_COLUMNS]
CATEGORICAL_FEATURES = ["State"]
train_df = model_df[model_df["Year"] <= 2021].copy()
test_df = model_df[model_df["Year"] >= 2022].copy()
if train_df.empty or test_df.empty:
    raise ValueError("Expected training records through 2021 and test records from 2022 onward.")
print("Original rows:", len(df))
print("Usable modelling rows:", len(model_df))
print("Training rows (through 2021):", len(train_df))
print("Testing rows (2022 onward):", len(test_df))
display(model_df.head())

## Chapter 4 — Model development
Use a chronological train/test split: historical records through 2021 train the model and 2022–2024 test it. Gradient Boosting learns the year-over-year capacity change; the predicted change is added to the previous year's total.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=2,
        min_samples_leaf=4, loss="huber", random_state=42
    )),
])
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train_change = train_df["Capacity Change"]
y_test_change = test_df["Capacity Change"]
y_test_total = test_df[TARGET].to_numpy()
model.fit(X_train, y_train_change)
predicted_change = model.predict(X_test)
previous_capacity = test_df[PREVIOUS_TOTAL].to_numpy()
model_predictions = np.maximum(previous_capacity + predicted_change, 0)
baseline_predictions = previous_capacity.copy()
print("Model training complete.")

## Chapter 5 — Results and evaluation
MAE and RMSE measure prediction error (lower is better). R² measures explained variation (closer to 1 is generally better). WAPE is total absolute error divided by total actual capacity. The persistence baseline assumes the next year's capacity equals the previous year's value.

In [ ]:
def calculate_metrics(actual, predicted, name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
        "WAPE (%)": np.abs(actual - predicted).sum() / max(np.abs(actual).sum(), 1e-9) * 100,
    }

metrics_df = pd.DataFrame([
    calculate_metrics(y_test_total, baseline_predictions, "Persistence baseline"),
    calculate_metrics(y_test_total, model_predictions, "Gradient Boosting"),
])
display(metrics_df.round(3))
metrics_df.to_csv("renewable_energy_model_metrics.csv", index=False)

predictions_df = test_df[["State", "Year", TARGET, PREVIOUS_TOTAL]].copy()
predictions_df["Predicted Capacity Change"] = predicted_change
predictions_df["Model Predicted Total Capacity"] = model_predictions
predictions_df["Persistence Baseline Prediction"] = baseline_predictions
predictions_df.to_csv("renewable_energy_backtest_predictions.csv", index=False)
model_df.to_csv("renewable_energy_preprocessed.csv", index=False)

plt.figure(figsize=(6, 5))
plt.scatter(y_test_total, model_predictions, alpha=.65)
limit = max(float(y_test_total.max()), float(model_predictions.max()))
plt.plot([0, limit], [0, limit], "--")
plt.xlabel("Actual capacity"); plt.ylabel("Predicted capacity")
plt.title("Actual vs Predicted Capacity (Test Set)")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "actual_vs_predicted.png", dpi=150); plt.show()

## Chapter 4 — Feature importance and model export
Permutation importance is a diagnostic showing how the model score changes when a feature is shuffled. It does not show causation. Finally, refit the chosen model on all eligible historical rows and save it.

In [ ]:
importance = permutation_importance(
    model, X_test, y_test_change, n_repeats=10,
    random_state=42, scoring="neg_mean_absolute_error", n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Mean importance": importance.importances_mean,
    "Importance standard deviation": importance.importances_std,
}).sort_values("Mean importance", ascending=False)
display(importance_df)
importance_df.to_csv("renewable_energy_feature_importance.csv", index=False)

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=2,
        min_samples_leaf=4, loss="huber", random_state=42
    )),
])
final_model.fit(model_df[FEATURES], model_df["Capacity Change"])
joblib.dump({
    "model": final_model,
    "feature_columns": FEATURES,
    "capacity_columns": CAPACITY_COLUMNS,
    "target_column": TARGET,
    "previous_total_column": PREVIOUS_TOTAL,
    "lag_columns": LAG_COLUMNS,
}, "renewable_energy_model.joblib")
print("Saved model, processed data, metrics, predictions and plots.")

## Chapter 3 — Git and GitHub
Git tracks versions of the project; GitHub hosts the repository online. After authenticating in your Colab session, you can clone, add, commit and push changes. Never paste a password or personal access token into a public notebook.

The connected account currently has this repository:
[`Renewable-energy-resoruces-in-India`](https://github.com/athmikathmik8-svg/Renewable-energy-resoruces-in-India). The repository name has a spelling difference from the project title; it can be renamed in GitHub settings if you need an exact name match.

In [ ]:
# Git workflow examples (run after authentication and cloning)
!git --version
# !git clone https://github.com/athmikathmik8-svg/Renewable-energy-resoruces-in-India.git
# %cd Renewable-energy-resoruces-in-India
# !git status
# !git add .
# !git commit -m "Update renewable energy analysis"
# !git push

## Chapter 5 — Conclusion and limitations
This project demonstrates data validation, lag-feature engineering, EDA, chronological model evaluation and comparison with a simple baseline. On the supplied test set, Gradient Boosting improves RMSE and R² relative to the persistence baseline but does not improve MAE and WAPE. Therefore it should not be described as better on every metric.

**Limitations:** the input CSV does not document capacity units or provenance; only 2022–2024 are held out; state-level estimates are not grid-engineering recommendations; and a multi-year future forecast requires future input values and a separately validated forecasting design.